# 3. A chaotic double pendulum

Two cubes joined by ball joints. Each body adds 30 unknowns per time step, so this example takes longer to compute than the others. Once computed, it plays smoothly.

In [ ]:
import sys
if sys.platform == "emscripten":  # the online version installs its packages here
    %pip install -q lib_rigid_body pythreejs

from netgen.occ import Box, Cylinder, Pnt
from lib_rigid_body import World
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def double_pendulum(nudge=0.0):
    cube = Box(Pnt(-.2, -.2, -.2), Pnt(.2, .2, .2))
    world = World()
    first = world.add(cube, at=(1, 0, 0), density=100)
    second = world.add(cube, at=(2, 0, nudge), density=100)
    world.pin(first, (0, 0, 0))
    world.ball(first, second, (1.5, 0, 0))
    return world.run(duration=5, h=0.01)

recording = double_pendulum()
print(recording.summary())
recording.show()

## Sensitivity to initial conditions

Run it again with the second cube one millimetre higher. The two motions separate quickly, while each one keeps its energy within a small band.

In [ ]:
nudged = double_pendulum(nudge=0.001)
separation = np.linalg.norm(recording.positions[:, 1] - nudged.positions[:, 1], axis=1)

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 2.8))
left.semilogy(recording.times, separation + 1e-16)
left.set(xlabel="time [s]", ylabel="separation [m]", title="Nearby motions diverge")
left.grid(alpha=.3)
recording.plot_energy(ax=right)
plt.tight_layout()